# Prepare the Twitter dataset

This notebook discovers every Twitter JSONL.GZ file in data/twitter and processes batches in filename order. It divides MAX_TWEETS evenly across batches and uses the selected sampling strategy for each batch. Add future daily archives to data/twitter; the notebook discovers them automatically.

SAMPLE_STRATEGY = "first" is fast: it stops reading each archive when its quota is met, but early rows may have no retweet-count variation. SAMPLE_STRATEGY = "reservoir" samples across each full archive and scans every source row. The current notebook setting uses reservoir sampling so the retweet-count regression target is more likely to vary.

## 1. Install dependencies

Run once per environment. The first embedding run also downloads Google's model weights from Hugging Face.

In [ ]:
from pathlib import Path
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "pyproject.toml").is_file() and (p / "imste").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate the IMSTE repository.")
%pip install -e "{PROJECT_ROOT}[notebook,preprocessing]"


## 2. Configure the sample and build embeddings

Set LANGUAGE to a Twitter language code (for example en) or None to accept all languages. MAX_TWEETS is the requested total across all batches. The notebook divides it evenly, giving any remainder to the earliest batches.

In [ ]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = 'retina'

from pathlib import Path
import sys
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "pyproject.toml").is_file() and (p / "imste").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate the IMSTE repository.")
for path in (PROJECT_ROOT, PROJECT_ROOT / "notebooks" / "src"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
DATA_DIR = PROJECT_ROOT / "data" / "twitter"
SOURCE_PATHS = sorted(DATA_DIR.glob("*.json.gz"), key=lambda path: path.name)
if not SOURCE_PATHS:
    raise FileNotFoundError(f"No Twitter JSON.GZ batches found in {DATA_DIR}")
print("Batch order:")
for path in SOURCE_PATHS:
    print(f"  - {path.name} ({path.stat().st_size / 1024**3:.2f} GiB compressed)")

Batch order:
  - 2014-12-01.world.json.gz (3.66 GiB compressed)
  - 2014-12-02.world.json.gz (3.54 GiB compressed)
  - 2014-12-03.world.json.gz (3.69 GiB compressed)


In [ ]:
from twitter_dataset import prepare_twitter_dataset
MAX_TWEETS = 1000_000
SAMPLE_STRATEGY = "reservoir"  # samples across each full archive; scans all source rows
RANDOM_STATE = 42
LANGUAGE = None                 # e.g. "en", or None for all languages
EMBEDDING_MODEL = "google/embeddinggemma-2"
BATCH_SIZE = 64
FORCE_REBUILD = False

tweets, X, manifest = prepare_twitter_dataset(
    DATA_DIR, SOURCE_PATHS, max_tweets=MAX_TWEETS,
    random_state=RANDOM_STATE, language=LANGUAGE,
    embedding_model=EMBEDDING_MODEL, batch_size=BATCH_SIZE,
    sample_strategy=SAMPLE_STRATEGY,
    force_rebuild=FORCE_REBUILD,
)
print(f"Prepared {len(tweets):,} tweets from {manifest['n_batches']} batches × {X.shape[1]:,} features")
print(f"Vector matrix: {X.nbytes / 1024**3:.2f} GiB")
print("Batch order:", manifest["batch_order"])
print("Sampling strategy:", manifest["sample_strategy"])
print("Batch quotas:", manifest["batch_quotas"])
print("Dataset ID:", manifest["dataset_id"])
tweets.head(10)